# Style Transfer API — Colab runner

Runs the prototype API inside Colab with ngrok so you can hit it from your browser.

In [ ]:
!git clone --depth 1 https://github.com/SaiAle/TensorFlow.git SaiAle-TensorFlow || true
!cp SaiAle-TensorFlow/FAST_STYLE_ARBITRARY_STYLES.ipynb /content/ || true
!mkdir -p StyleTransfer-Prototype/src tests var
%%writefile StyleTransfer-Prototype/pyproject.toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "style-transfer-prototype"
version = "0.1.0"
description = "Prototype built from Fast Neural Arbitrary Style Transfer"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
    "fastapi>=0.109",
    "tensorflow>=2.15",
    "tensorflow-hub",
    "pillow",
    "python-multipart",
    "uvicorn[standard]",
]

[project.optional-dependencies]
dev = [
    "pytest>=7",
    "pytest-cov",
    "ruff",
]

[project.scripts]
stylize = "src.cli:main"
style-api = "src.api_serve:main"

[tool.hatch.build.targets.wheel]
packages = ["src"]

In [ ]:
%%writefile StyleTransfer-Prototype/src/__init__.py



In [ ]:
%%writefile StyleTransfer-Prototype/src/models.py
from __future__ import annotations

DEFAULT_MODEL = "https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2"

FALLBACK_MODELS: list[dict[str, str]] = [
    {"alias": "magenta", "url": DEFAULT_MODEL},
]


def resolve(url: str | None = None, alias: str | None = None) -> str:
    if url:
        return url
    if alias:
        for model in FALLBACK_MODELS:
            if model["alias"] == alias:
                return model["url"]
    return DEFAULT_MODEL

In [ ]:
%%writefile StyleTransfer-Prototype/src/utils.py
from __future__ import annotations

from pathlib import Path


def ensure(path: Path) -> Path:
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)

In [ ]:
%%writefile StyleTransfer-Prototype/src/caching.py
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any


def cache_path(cache_dir: Path, content_key: str) -> Path:
    digest = hashlib.sha256(content_key.encode()).hexdigest()[:16]
    return cache_dir / f"{digest}.json"


def load_cache(path: Path) -> Any | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except Exception:
        return None


def save_cache(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload))

In [ ]:
%%writefile StyleTransfer-Prototype/src/inference.py
from __future__ import annotations

import io
import sys
import types
from pathlib import Path
from typing import Union

try:
    import numpy as np
except Exception:
    np = None  # type: ignore[assignment]

from PIL import Image

try:
    import tensorflow as tf
except Exception as _tf_exc:
    raise RuntimeError(f"tensorflow is required: {_tf_exc}") from _tf_exc

# TensorFlow Hub in Colab generally imports cleanly; keep a fallback just in case.
try:
    import tensorflow_hub as hub
except Exception:
    hub = None  # type: ignore[assignment]

from .models import resolve


def _decode(content: bytes) -> "np.ndarray":
    img = Image.open(io.BytesIO(content)).convert("RGB")
    return np.array(img)


def _encode_png(array: "np.ndarray") -> bytes:
    img = Image.fromarray(np.clip(array, 0, 255).astype(np.uint8))
    out = io.BytesIO()
    img.save(out, format="PNG")
    return out.getvalue()


def stylize_bytes(
    content: bytes,
    style: bytes,
    model_url: str,
    *,
    preserve_aspect_ratio: bool = True,
) -> bytes:
    if hub is None:
        raise RuntimeError("tensorflow-hub is not installed in this environment")

    content_image = _decode(content)
    style_image = _decode(style)

    hub_module = hub.load(model_url)

    content_tensor = tf.constant(content_image[np.newaxis, ...], dtype=tf.float32)
    style_tensor = tf.constant(style_image[np.newaxis, ...], dtype=tf.float32)

    outputs = hub_module(content_tensor, style_tensor)
    result = outputs[0][0].numpy()

    return _encode_png(result)

In [ ]:
%%writefile StyleTransfer-Prototype/src/core.py
from __future__ import annotations

from pathlib import Path
from typing import Union

from .caching import cache_path, load_cache, save_cache
from .inference import stylize_bytes
from .models import resolve

try:
    import numpy as np
    from PIL import Image
except Exception:
    np = None
    Image = None


def apply_style(
    content_path: Path,
    style_path: Path,
    out_path: Path,
    model_url: str,
    *,
    preserve_aspect_ratio: bool = True,
    cache_dir: Path | None = None,
) -> dict:
    if np is None or Image is None:
        raise RuntimeError("numpy and Pillow are required")

    content_bytes = content_path.read_bytes()
    style_bytes = style_path.read_bytes()

    cache_material = (
        f"{model_url}|{content_path}:{len(content_bytes)}|{style_path}:{len(style_bytes)}"
    )
    cached = None
    if cache_dir is not None:
        cached = load_cache(cache_path(cache_dir, cache_material))
    if cached is not None:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_bytes(cached.get("png", b""))
        return _summary(out_path, used_cache=True)

    png_bytes = stylize_bytes(content_bytes, style_bytes, model_url)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(png_bytes)

    if cache_dir is not None:
        save_cache(cache_path(cache_dir, cache_material), {"png": png_bytes})

    return _summary(out_path, used_cache=False)


def tensor_to_image(tensor) -> "Image.Image":
    try:
        array = tensor.numpy()
    except AttributeError:
        array = np.asarray(tensor)

    if np.max(array) <= 1:
        array = array * 255
    array = np.clip(array, 0, 255).astype("uint8")
    return Image.fromarray(array)


def save_tensor(path: Path, tensor) -> None:
    tensor_to_image(tensor).save(path)


def _summary(path: Path, *, used_cache: bool) -> dict:
    return {
        "out": str(path),
        "bytes": path.stat().st_size,
        "cached": used_cache,
    }

In [ ]:
%%writefile StyleTransfer-Prototype/src/cli.py
from __future__ import annotations

import argparse
from pathlib import Path

from .core import apply_style
from .utils import ensure
from .models import resolve


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="stylize")
    p.add_argument("content", type=Path, help="Content image path")
    p.add_argument("style", type=Path, help="Style image path")
    p.add_argument("--out", type=Path, default=Path("out.png"), help="Output path")
    p.add_argument("--model", default=None, help="TF Hub model URL or alias")
    p.add_argument("--cache-dir", type=Path, default=None, help="Cache directory")
    p.add_argument("--list-models", action="store_true", help="List fallback models")
    return p


def main(argv: list[str] | None = None) -> int:
    from .models import FALLBACK_MODELS, DEFAULT_MODEL

    args = build_parser().parse_args(argv)
    if args.list_models:
        print("Default:", DEFAULT_MODEL)
        for m in FALLBACK_MODELS:
            print("-", m["alias"], m["url"])
        return 0

    if not args.content.exists() or not args.style.exists():
        raise SystemExit("content and style must exist")

    cache_dir = ensure(args.cache_dir) if args.cache_dir else None
    model_url = resolve(args.model)

    summary = apply_style(
        args.content,
        args.style,
        args.out,
        model_url=model_url,
        cache_dir=cache_dir,
    )
    print(summary)
    return 0

In [ ]:
%%writefile StyleTransfer-Prototype/src/worker.py
from __future__ import annotations

import json
from pathlib import Path

from .core import apply_style
from .utils import ensure
from .models import resolve


def run_batch(
    content_dir: Path,
    style_dir: Path,
    out_dir: Path,
    model_url: str,
    *,
    cache_dir: Path | None = None,
    pattern: str = "*.png,*.jpg,*.jpeg",
) -> list[dict]:
    ensure(content_dir)
    ensure(style_dir)
    ensure(out_dir)

    content_paths = _resolve(content_dir, pattern)
    style_paths = _resolve(style_dir, pattern)

    if not content_paths:
        raise ValueError(f"no content files under {content_dir}")
    if not style_paths:
        raise ValueError(f"no style files under {style_dir}")

    if cache_dir is not None:
        ensure(cache_dir)

    records: list[dict] = []

    for cp in content_paths:
        for sp in style_paths:
            out = out_dir / f"{cp.stem}__{sp.stem}.png"
            summary = apply_style(
                cp,
                sp,
                out,
                model_url=model_url,
                cache_dir=cache_dir,
            )
            records.append(summary)

    manifest = out_dir / "manifest.json"
    manifest.write_text(json.dumps(records, indent=2))
    return records


def _resolve(base: Path, pattern: str):
    matches: list[Path] = []
    for leaf in pattern.split(","):
        matches.extend(sorted(base.rglob(leaf.strip())))
    return matches

In [ ]:
%%writefile StyleTransfer-Prototype/src/api.py
from __future__ import annotations

from pathlib import Path
from typing import Literal

from fastapi import Depends, FastAPI, File, HTTPException, UploadFile
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel

from .core import apply_style
from .utils import ensure
from .models import resolve


class TransferResponse(BaseModel):
    out: str
    bytes: int
    cached: bool


class PresetIn(BaseModel):
    name: str
    style_rel_path: str


class PresetOut(BaseModel):
    id: str
    name: str
    style_rel_path: str


class TransferByPresetRequest(BaseModel):
    content_filename: str
    preset_id: str
    model: str | None = None
    cache: bool = False


def _presets_path(data_dir: Path) -> Path:
    return data_dir / "presets.json"


def _load_presets(path: Path) -> dict[str, PresetOut]:
    if not path.exists():
        return {}
    import json

    raw = json.loads(path.read_text())
    out: dict[str, PresetOut] = {}
    for item in raw:
        out[item["id"]] = PresetOut(**item)
    return out


def _save_presets(path: Path, presets: dict[str, PresetOut]) -> None:
    import json

    path.write_text(json.dumps([v.model_dump() for v in presets.values()], indent=2))


def create_app(data_dir: Path) -> FastAPI:
    app = FastAPI(title="Style Transfer")
    presets_file = _presets_path(data_dir)

    @app.get("/")
    def read_root() -> dict[str, str]:
        return {"service": "style-transfer-api", "status": "ok"}

    @app.get("/health")
    def health() -> dict[str, str]:
        return {"status": "ok", "data_dir": str(data_dir)}

    @app.post("/transfer", response_model=TransferResponse)
    async def transfer(
        content: UploadFile = File(...),
        style: UploadFile = File(...),
        model: str | None = None,
        cache: bool = False,
    ) -> TransferResponse:
        try:
            content_path = data_dir / content.filename
            style_path = data_dir / style.filename
            content_bytes = await content.read()
            style_bytes = await style.read()
            content_path.write_bytes(content_bytes)
            style_path.write_bytes(style_bytes)
        except Exception as exc:
            raise HTTPException(400, f"invalid upload: {exc}") from exc

        out_path = data_dir / f"out_{content_path.stem}_{style_path.stem}.png"
        cache_dir = ensure(data_dir / ".cache") if cache else None

        summary = apply_style(
            content_path,
            style_path,
            out_path,
            model_url=resolve(model),
            cache_dir=cache_dir,
        )
        return TransferResponse(**summary)

    @app.post("/transfer/preset", response_model=TransferResponse)
    async def transfer_by_preset(payload: TransferByPresetRequest) -> TransferResponse:
        presets = _load_presets(presets_file)
        preset = presets.get(payload.preset_id)
        if preset is None:
            raise HTTPException(404, f"unknown preset_id: {payload.preset_id}")

        content_path = (data_dir / payload.content_filename).resolve()
        style_path = (data_dir / preset.style_rel_path).resolve()

        if not content_path.exists():
            raise HTTPException(404, f"missing content file: {payload.content_filename}")
        if not style_path.exists():
            raise HTTPException(500, f"preset style file missing: {preset.style_rel_path}")
        if not str(content_path).startswith(str(data_dir.resolve())):
            raise HTTPException(400, "content_filename must stay inside data_dir")

        out_path = data_dir / f"out_{content_path.stem}__{preset.id}.png"
        cache_dir = ensure(data_dir / ".cache") if payload.cache else None

        summary = apply_style(
            content_path,
            style_path,
            out_path,
            model_url=resolve(payload.model),
            cache_dir=cache_dir,
        )
        return TransferResponse(**summary)

    @app.post("/presets", response_model=PresetOut)
    async def create_preset(payload: PresetIn) -> PresetOut:
        import uuid

        style_path = (data_dir / payload.style_rel_path).resolve()
        if not style_path.exists():
            raise HTTPException(400, f"missing style file: {payload.style_rel_path}")
        if not str(style_path).startswith(str(data_dir.resolve())):
            raise HTTPException(400, "style_rel_path must stay inside data_dir")

        preset_id = uuid.uuid4().hex[:8]
        preset = PresetOut(id=preset_id, name=payload.name, style_rel_path=payload.style_rel_path)
        presets = _load_presets(presets_file)
        presets[preset_id] = preset
        _save_presets(presets_file, presets)
        return preset

    @app.get("/presets", response_model=list[PresetOut])
    def list_presets() -> list[PresetOut]:
        return list(_load_presets(presets_file).values())

    return app

In [ ]:
%%writefile StyleTransfer-Prototype/src/api_serve.py
from __future__ import annotations

import os
from pathlib import Path

from .api import create_app


def main(argv: list[str] | None = None) -> int:
    import argparse

    p = argparse.ArgumentParser()
    p.add_argument("--host", default="127.0.0.1")
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--data-dir", type=Path, default=None)
    p.add_argument("--reload", action="store_true")
    args = p.parse_args(argv)

    data_dir = args.data_dir or Path(os.getcwd()) / "var"
    data_dir.mkdir(parents=True, exist_ok=True)

    import uvicorn

    uvicorn.run(
        create_app(data_dir),
        host=args.host,
        port=args.port,
        reload=args.reload,
    )
    return 0

In [ ]:
# Install dependencies
!pip install -q fastapi uvicorn python-multipart pillow "tensorflow>=2.15" tensorflow-hub

# Start API in background
!mkdir -p StyleTransfer-Prototype/var
%cd StyleTransfer-Prototype
!python -m uvicorn src.api_serve:create_app --factory --host 0.0.0.0 --port 8000 &

# Install ngrok-free and expose it
!pip install -q pyngrok
from pyngrok import ngrok
url = ngrok.connect(8000)
print("Public API:", url)